[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/28_moe_solution.ipynb)

# ✅ Solution: moe

Implement a **Mixture of Experts** layer (Mixtral / Switch Transformer style).

### Signature
```python
class MixtureOfExperts(nnx.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2): ...
    def forward(self, x: Tensor) -> Tensor:
        # x: (B, S, D) -> (B, S, D)
```

### Architecture
- `self.router`: `nn.Linear(d_model, num_experts)` — gating network
- `self.experts`: `nn.ModuleList` of MLPs `(Linear→ReLU→Linear)`
- For each token: select top-k experts, compute weighted sum of their outputs


In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge')
except ImportError:
    pass


In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx
import math


In [ ]:
# ✅ SOLUTION

import jax, jax.numpy as jnp
from flax import nnx
class _Expert(nnx.Module):
    def __init__(self, d, f, *, rngs):
        self.fc1 = nnx.Linear(d, f, rngs=rngs)
        self.fc2 = nnx.Linear(f, d, rngs=rngs)
    def __call__(self, x_ND):
        return self.fc2(jax.nn.relu(self.fc1(x_ND)))
class MixtureOfExperts(nnx.Module):
    """B=batch, L=seq, D=d_model, N=B*L tokens, E=experts."""
    def __init__(self, d_model, d_ff, num_experts, top_k=2, *, rngs):
        self.top_k = top_k
        self.router = nnx.Linear(d_model, num_experts, rngs=rngs)
        self.experts = nnx.List([_Expert(d_model, d_ff, rngs=rngs) for _ in range(num_experts)])
    def __call__(self, x_BLD):
        shape_BLD = x_BLD.shape
        x_ND = x_BLD.reshape(-1, shape_BLD[-1])
        logits_NE = self.router(x_ND)
        vals_NK, idx_NK = jax.lax.top_k(logits_NE, self.top_k)
        weights_NK = jax.nn.softmax(vals_NK, axis=-1)
        out_ND = jnp.zeros_like(x_ND)
        for k in range(self.top_k):
            for e, expert in enumerate(self.experts):
                contrib_ND = weights_NK[:, k : k + 1] * expert(x_ND)
                out_ND = out_ND + jnp.where(idx_NK[:, k : k + 1] == e, contrib_ND, 0)
        return out_ND.reshape(shape_BLD)


In [ ]:
# Verify
print(MixtureOfExperts)


In [ ]:
from jax_judge import check
check("moe")
